In [ ]:
#!/usr/bin/env python3

import v4l2
from fcntl import ioctl
import glob, errno
# import pprint

In [ ]:
def ctrl_type_str(ctrl):
    return {
        v4l2.V4L2_CTRL_TYPE_INTEGER: "INTEGER",
        v4l2.V4L2_CTRL_TYPE_BOOLEAN: "BOOLEAN",
        v4l2.V4L2_CTRL_TYPE_MENU: "MENU",
        v4l2.V4L2_CTRL_TYPE_BUTTON: "BUTTON",
        v4l2.V4L2_CTRL_TYPE_INTEGER64: "INTEGER64",
        v4l2.V4L2_CTRL_TYPE_CTRL_CLASS: "CTRL_CLASS",
        v4l2.V4L2_CTRL_TYPE_STRING: "STRING",
        v4l2.V4L2_CTRL_TYPE_BITMASK: "BITMASK",
        v4l2.V4L2_CTRL_TYPE_INTEGER_MENU: "INTEGER_MENU",
    }[ctrl.type]

def get_ctrl_value(dev, ctrl):
    gctrl = v4l2.v4l2_control()
    gctrl.id = ctrl.id

    try:
        ioctl(dev, v4l2.VIDIOC_G_CTRL, gctrl)
    except OSError:
        return None

    return gctrl.value

def set_ctrl_value(dev, ctrl, value):
    sctrl = v4l2.v4l2_control()
    sctrl.id = ctrl.id
    sctrl.value = value

    try:
        ioctl(dev, v4l2.VIDIOC_S_CTRL, sctrl)
    except OSError:
        # can fail as some controls can be read-only
        # both explicitly (by setting flag) or implicitly
        return

def is_valid_device(device):
    ctrl = v4l2.v4l2_queryctrl()
    ctrl.id = v4l2.V4L2_CTRL_FLAG_NEXT_CTRL
    try:
        ioctl(device, v4l2.VIDIOC_QUERYCTRL, ctrl)
    except OSError as e:
        # error code returned when device
        # gets disconnected
        return e.errno != errno.ENODEV

    return True

def query_v4l2_ctrls(dev):
    ctrl_id = v4l2.V4L2_CTRL_FLAG_NEXT_CTRL
    current_class = "User Controls"
    controls = {current_class: []}

    while True:
        ctrl = v4l2.v4l2_query_ext_ctrl()
        ctrl.id = ctrl_id
        try:
            ioctl(dev, v4l2.VIDIOC_QUERY_EXT_CTRL, ctrl)
        except OSError:
            # we reached last control
            break

        if ctrl.type == v4l2.V4L2_CTRL_TYPE_CTRL_CLASS:
            current_class = ctrl.name.decode("utf-8")
            controls[current_class] = []

        controls[current_class].append(ctrl)

        ctrl_id = ctrl.id | v4l2.V4L2_CTRL_FLAG_NEXT_CTRL

    return controls

def query_tegra_ctrls(dev):
    """This function supports deprecated TEGRA_CAMERA_CID_* API"""
    ctrls = []

    ctrlid = v4l2.TEGRA_CAMERA_CID_BASE

    ctrl = v4l2.v4l2_queryctrl()
    ctrl.id = ctrlid

    while ctrl.id < v4l2.TEGRA_CAMERA_CID_LASTP1:
        try:
            ioctl(dev, v4l2.VIDIOC_QUERYCTRL, ctrl)
        except IOError as e:
            if e.errno != errno.EINVAL:
                break
            ctrl = v4l2.v4l2_queryctrl()
            ctrlid += 1
            ctrl.id = ctrlid
            continue

        if not ctrl.flags & v4l2.V4L2_CTRL_FLAG_DISABLED:
            ctrls.append(ctrl)

        ctrl = v4l2.v4l2_queryctrl()
        ctrlid += 1
        ctrl.id = ctrlid

    return {"Tegra Controls": ctrls}


def query_ctrls(dev):
    ctrls_v4l2 = query_v4l2_ctrls(dev)
    ctrls_tegra = query_tegra_ctrls(dev)

    return {**ctrls_v4l2, **ctrls_tegra}


def query_driver(dev):
    try:
        cp = v4l2.v4l2_capability()
        ioctl(dev, v4l2.VIDIOC_QUERYCAP, cp)
        return cp.driver
    except Exception:
        return b"unknown"

In [ ]:
gs_dev = glob.glob('/dev/links/gs*')
dev = open(gs_dev[0], "rb")

try:
    ctrls = query_ctrls(dev)
    if len(sum(ctrls.values(), start=[])) != 0:
   
        for name, controls in ctrls.items():
            for ctrl in controls:
                try:
                    value = get_ctrl_value(dev, ctrl)
                    if ctrl.type == v4l2.V4L2_CTRL_TYPE_CTRL_CLASS:
                        print(f"{ctrl.name.decode('utf-8')}:::>")
                    else:
                        print(f"    {ctrl.name.decode('utf-8')}: {value}, {ctrl_type_str(ctrl)}")
                        if ctrl.type == v4l2.V4L2_CTRL_TYPE_INTEGER:
                            print(f"        min-max-step: {ctrl.minimum} - {ctrl.maximum}, step: {ctrl.step}")
                except KeyError:
                    # XXX quietly skip unsupported control types
                    continue
        
        all_controls = {p.name.decode('utf-8'):p for name,cls in ctrls.items() for p in cls}
        # pprint.pprint(all_controls)
        # try set brightness control to 1280
    
        if "Brightness" in all_controls:
            set_ctrl_value(dev, all_controls["Brightness"], 0)
            brtns = get_ctrl_value(dev, all_controls["Brightness"])
            print(f"Brightness set to {brtns}")

finally:
    dev.close()